# 03 Data Cleaning

Applies the corrections recommended in `02_data_quality.ipynb` and publishes a cleaned layer next to the raw tables.

| Table | Built from | Grain |
| --- | --- | --- |
| `train_run_events_clean` | `train_run_events` | One row per train run per station stop |
| `train_master_clean` | `train_master` | One row per train |
| `station_master_clean` | `station_master` | One row per station |

Raw tables and CSV files are not modified. Idempotent full refresh. Run notebook 01 first.

## 1. Setup

Read-write connection, because this notebook publishes tables. Fails fast if the raw tables are missing.

In [1]:
from railpulse.config import DB_PATH, PROJECT_ROOT, RAW_TABLES
from railpulse.db import connect

con = connect(read_only=False, require=RAW_TABLES)


def show(sql, max_rows=50):
    con.sql(sql).show(max_width=220, max_rows=max_rows)


# fingerprint the raw tables so section 5 can check they weren't changed
RAW_FINGERPRINT_SQL = """
    SELECT 'train_run_events' AS table_name, COUNT(*) AS row_count, bit_xor(hash(t)) AS fingerprint
    FROM train_run_events t
    UNION ALL
    SELECT 'train_master', COUNT(*), bit_xor(hash(t)) FROM train_master t
    UNION ALL
    SELECT 'station_master', COUNT(*), bit_xor(hash(t)) FROM station_master t
    ORDER BY table_name
"""
raw_before = con.sql(RAW_FINGERPRINT_SQL).fetchall()

print(f"Connected to {DB_PATH.relative_to(PROJECT_ROOT)}")

Connected to railpulse.duckdb


## 2. Corrections applied

| # | Issue from notebook 02 | Correction | Where |
| --- | --- | --- | --- |
| 1 | Stop position is implicit | `is_origin`, `is_terminus` flags | events |
| 2 | Origin arrival is a placeholder | Arrival fields set to `NULL` at the origin | events |
| 3 | Terminus departure is not an observation | Departure fields set to `NULL` at the terminus | events |
| 4 | Negative dwell | `has_negative_dwell` flag, dwell set to `NULL`, row kept | events |
| 5 | Dwell is meaningless at route ends | Dwell populated on intermediate stops only | events |
| 6 | Early running offsets lateness | Signed delay kept, `*_late_min` added | events |
| 7 | Large delays need to be identifiable | `is_large_*_delay` flags from the Tukey upper fence | events |
| 8 | Names repeated in the events | Name columns dropped, master tables are the source | events |
| 9 | Unserved station `ST006` | `is_served` flag | stations |
| 10 | Weekly day off looks like missing runs | `runs_operated`, `non_operating_weekday` | trains |

## 3. Build `train_run_events_clean`

One statement, three steps: position flags, placeholder nulling, derived measures. Every raw row is kept.

In [2]:
con.execute("""
    CREATE OR REPLACE TABLE train_run_events_clean AS
    WITH positioned AS (
        SELECT
            *,
            -- same origin/terminus logic as notebook 02
            station_sequence = MIN(station_sequence) OVER run AS is_origin,
            station_sequence = MAX(station_sequence) OVER run AS is_terminus
        FROM train_run_events
        WINDOW run AS (PARTITION BY service_id)
    ),
    observed AS (
        -- null out the placeholder arrivals/departures so AVG etc. skip them
        -- names are dropped, take them from the master tables
        SELECT
            service_id,
            service_date,
            train_id,
            station_sequence,
            station_id,
            is_origin,
            is_terminus,
            CASE WHEN NOT is_origin THEN scheduled_arrival END AS scheduled_arrival,
            CASE WHEN NOT is_origin THEN actual_arrival END AS actual_arrival,
            CASE WHEN NOT is_origin THEN arrival_delay_min END AS arrival_delay_min,
            CASE WHEN NOT is_terminus THEN scheduled_departure END AS scheduled_departure,
            CASE WHEN NOT is_terminus THEN actual_departure END AS actual_departure,
            CASE WHEN NOT is_terminus THEN departure_delay_min END AS departure_delay_min,
            simulated_disruption_flag,
            simulated_disruption_type,
            data_status
        FROM positioned
    ),
    fences AS (
        -- tukey fence, computed after the nulling so placeholders don't skew it
        SELECT
            quantile_cont(arrival_delay_min, 0.75) + 1.5 * (
                quantile_cont(arrival_delay_min, 0.75) - quantile_cont(arrival_delay_min, 0.25)
            ) AS arrival_fence_min,
            quantile_cont(departure_delay_min, 0.75) + 1.5 * (
                quantile_cont(departure_delay_min, 0.75) - quantile_cont(departure_delay_min, 0.25)
            ) AS departure_fence_min
        FROM observed
    )
    SELECT
        o.service_id,
        o.service_date,
        o.train_id,
        o.station_sequence,
        o.station_id,
        o.is_origin,
        o.is_terminus,
        o.scheduled_arrival,
        o.actual_arrival,
        o.arrival_delay_min,
        o.scheduled_departure,
        o.actual_departure,
        o.departure_delay_min,
        -- early counts as 0 late. need the CASE because GREATEST(NULL, 0) returns 0
        CASE WHEN o.arrival_delay_min IS NOT NULL THEN GREATEST(o.arrival_delay_min, 0) END AS arrival_late_min,
        CASE WHEN o.departure_delay_min IS NOT NULL THEN GREATEST(o.departure_delay_min, 0) END AS departure_late_min,
        -- null at origin/terminus since one side was nulled above
        date_diff('minute', o.scheduled_arrival, o.scheduled_departure) AS scheduled_dwell_min,
        -- negative dwell becomes null (flagged below) rather than 0
        CASE
            WHEN o.actual_departure >= o.actual_arrival
            THEN date_diff('minute', o.actual_arrival, o.actual_departure)
        END AS actual_dwell_min,
        COALESCE(o.actual_departure < o.actual_arrival, FALSE) AS has_negative_dwell,
        COALESCE(o.arrival_delay_min > f.arrival_fence_min, FALSE) AS is_large_arrival_delay,
        COALESCE(o.departure_delay_min > f.departure_fence_min, FALSE) AS is_large_departure_delay,
        o.simulated_disruption_flag,
        o.simulated_disruption_type,
        o.data_status
    FROM observed o
    CROSS JOIN fences f
    ORDER BY o.service_id, o.station_sequence
""")
print("Created table: train_run_events_clean")

Created table: train_run_events_clean


## 4. Build the cleaned master tables

### 4.1 `station_master_clean`

Adds `is_served` so an unserved station is labelled instead of showing up as an unexplained empty row.

In [3]:
con.execute("""
    CREATE OR REPLACE TABLE station_master_clean AS
    SELECT
        s.station_id,
        s.station_name,
        s.region,
        -- COUNT(e.col) so unserved stations get 0
        COUNT(e.station_id) AS stop_events,
        COUNT(e.station_id) > 0 AS is_served
    FROM station_master s
    LEFT JOIN train_run_events e ON e.station_id = s.station_id
    GROUP BY s.station_id, s.station_name, s.region
    ORDER BY s.station_id
""")
print("Created table: station_master_clean")

Created table: station_master_clean


### 4.2 `train_master_clean`

Resolves endpoint names to station IDs and adds the operated-run count, the correct denominator for per-train rates.

In [4]:
con.execute("""
    CREATE OR REPLACE TABLE train_master_clean AS
    WITH runs AS (
        SELECT train_id, service_id, service_date, COUNT(*) AS stops
        FROM train_run_events
        GROUP BY train_id, service_id, service_date
    ),
    run_stats AS (
        SELECT
            train_id,
            COUNT(*) AS runs_operated,
            MAX(stops) AS stops_per_run,
            MIN(service_date) AS first_service_date,
            MAX(service_date) AS last_service_date
        FROM runs
        GROUP BY train_id
    ),
    calendar AS (
        -- every date in the period
        SELECT UNNEST(generate_series(
            (SELECT MIN(service_date) FROM train_run_events),
            (SELECT MAX(service_date) FROM train_run_events),
            INTERVAL 1 DAY
        ))::DATE AS service_date
    ),
    days_off AS (
        SELECT
            t.train_id,
            COUNT(*) AS days_without_a_run,
            -- one weekday = regular day off, more than one = gaps in the data
            string_agg(DISTINCT dayname(c.service_date), ', ') AS non_operating_weekday
        FROM train_master t
        CROSS JOIN calendar c
        LEFT JOIN runs r ON r.train_id = t.train_id AND r.service_date = c.service_date
        WHERE r.service_id IS NULL
        GROUP BY t.train_id
    )
    SELECT
        t.train_id,
        t.train_name,
        -- endpoint ids so later joins don't rely on names
        o.station_id AS origin_station_id,
        t.origin_station,
        d.station_id AS destination_station_id,
        t.destination_station,
        t.service_type,
        t.route_note,
        COALESCE(rs.stops_per_run, 0) AS stops_per_run,
        COALESCE(rs.runs_operated, 0) AS runs_operated,
        rs.first_service_date,
        rs.last_service_date,
        COALESCE(x.days_without_a_run, 0) AS days_without_a_run,
        x.non_operating_weekday
    FROM train_master t
    LEFT JOIN station_master o ON o.station_name = t.origin_station
    LEFT JOIN station_master d ON d.station_name = t.destination_station
    LEFT JOIN run_stats rs ON rs.train_id = t.train_id
    LEFT JOIN days_off x ON x.train_id = t.train_id
    ORDER BY t.train_id
""")
print("Created table: train_master_clean")

Created table: train_master_clean


## 5. Validate the cleaned layer

### 5.1 Before and after

Each issue from notebook 02, counted in the raw table and again in the cleaned table. Pass: `clean` is 0 on every row.

In [5]:
show("""
    WITH raw AS (
        SELECT
            *,
            station_sequence = MIN(station_sequence) OVER run AS is_origin,
            station_sequence = MAX(station_sequence) OVER run AS is_terminus
        FROM train_run_events
        WINDOW run AS (PARTITION BY service_id)
    )
    SELECT 1 AS check_no, 'Origin rows carrying an arrival' AS issue,
        (SELECT COUNT(*) FROM raw WHERE is_origin AND actual_arrival IS NOT NULL) AS raw,
        (SELECT COUNT(*) FROM train_run_events_clean WHERE is_origin AND
            (scheduled_arrival IS NOT NULL OR actual_arrival IS NOT NULL OR arrival_delay_min IS NOT NULL)) AS clean
    UNION ALL
    SELECT 2, 'Terminus rows carrying a departure',
        (SELECT COUNT(*) FROM raw WHERE is_terminus AND actual_departure IS NOT NULL),
        (SELECT COUNT(*) FROM train_run_events_clean WHERE is_terminus AND
            (scheduled_departure IS NOT NULL OR actual_departure IS NOT NULL OR departure_delay_min IS NOT NULL))
    UNION ALL
    SELECT 3, 'Negative dwell values',
        (SELECT COUNT(*) FROM raw WHERE actual_departure < actual_arrival),
        (SELECT COUNT(*) FROM train_run_events_clean WHERE actual_dwell_min < 0)
    UNION ALL
    SELECT 4, 'Dwell values at origin or terminus',
        (SELECT COUNT(*) FROM raw WHERE is_origin OR is_terminus),
        (SELECT COUNT(*) FROM train_run_events_clean WHERE (is_origin OR is_terminus) AND
            (actual_dwell_min IS NOT NULL OR scheduled_dwell_min IS NOT NULL))
    UNION ALL
    SELECT 5, 'Negative dwell rows without a flag',
        (SELECT COUNT(*) FROM raw WHERE actual_departure < actual_arrival AND NOT is_origin AND NOT is_terminus),
        (SELECT COUNT(*) FROM train_run_events_clean
         WHERE actual_departure < actual_arrival AND NOT has_negative_dwell)
    UNION ALL
    SELECT 6, 'Name columns repeated in the events',
        (SELECT COUNT(*) FROM information_schema.columns
         WHERE table_name = 'train_run_events' AND column_name IN ('train_name', 'station_name')),
        (SELECT COUNT(*) FROM information_schema.columns
         WHERE table_name = 'train_run_events_clean' AND column_name IN ('train_name', 'station_name'))
    UNION ALL
    SELECT 7, 'Unserved stations without a label',
        (SELECT COUNT(*) FROM station_master s
         LEFT JOIN train_run_events e ON e.station_id = s.station_id WHERE e.station_id IS NULL),
        (SELECT COUNT(*) FROM station_master_clean WHERE stop_events = 0 AND is_served)
    UNION ALL
    SELECT 8, 'Trains with a day off and no recorded weekday',
        (SELECT COUNT(*) FROM (
            SELECT train_id FROM train_run_events GROUP BY train_id
            HAVING COUNT(DISTINCT service_date) < (
                SELECT date_diff('day', MIN(service_date), MAX(service_date)) + 1 FROM train_run_events))),
        (SELECT COUNT(*) FROM train_master_clean
         WHERE days_without_a_run > 0 AND non_operating_weekday IS NULL)
    ORDER BY check_no
""")

┌──────────┬───────────────────────────────────────────────┬───────┬───────┐
│ check_no │                     issue                     │  raw  │ clean │
│  int32   │                    varchar                    │ int64 │ int64 │
├──────────┼───────────────────────────────────────────────┼───────┼───────┤
│        1 │ Origin rows carrying an arrival               │   426 │     0 │
│        2 │ Terminus rows carrying a departure            │   426 │     0 │
│        3 │ Negative dwell values                         │    42 │     0 │
│        4 │ Dwell values at origin or terminus            │   852 │     0 │
│        5 │ Negative dwell rows without a flag            │    13 │     0 │
│        6 │ Name columns repeated in the events           │     2 │     0 │
│        7 │ Unserved stations without a label             │     1 │     0 │
│        8 │ Trains with a day off and no recorded weekday │     4 │     0 │
└──────────┴───────────────────────────────────────────────┴───────┴───────┘

### 5.2 Integrity of the cleaned layer

Cleaning must not lose rows, break keys or alter a genuine observation. Assertions stop the notebook on any violation.

In [6]:
checks = con.sql("""
    SELECT
        (SELECT COUNT(*) FROM train_run_events) AS raw_rows,
        (SELECT COUNT(*) FROM train_run_events_clean) AS clean_rows,
        (SELECT COUNT(*) - COUNT(DISTINCT (service_id, station_sequence)) FROM train_run_events_clean) AS duplicate_keys,
        -- real observations should match raw exactly
        (SELECT COUNT(*)
         FROM train_run_events_clean c
         JOIN train_run_events r USING (service_id, station_sequence)
         WHERE (NOT c.is_origin AND (c.actual_arrival <> r.actual_arrival OR c.arrival_delay_min <> r.arrival_delay_min))
            OR (NOT c.is_terminus AND (c.actual_departure <> r.actual_departure OR c.departure_delay_min <> r.departure_delay_min))
        ) AS altered_observations,
        (SELECT COUNT(*) FROM train_run_events_clean c
         LEFT JOIN train_master_clean t USING (train_id)
         LEFT JOIN station_master_clean s USING (station_id)
         WHERE t.train_id IS NULL OR s.station_id IS NULL) AS orphan_rows,
        (SELECT COUNT(*) FROM train_run_events_clean
         WHERE arrival_late_min < 0 OR departure_late_min < 0) AS negative_lateness,
        (SELECT COUNT(*) FROM train_master_clean
         WHERE origin_station_id IS NULL OR destination_station_id IS NULL) AS unresolved_endpoints
""")
checks.show(max_width=220)

raw_rows, clean_rows, duplicate_keys, altered, orphans, negative_lateness, unresolved = checks.fetchone()
assert clean_rows == raw_rows, f"Row count changed: {raw_rows} raw, {clean_rows} clean"
assert duplicate_keys == 0, f"{duplicate_keys} duplicate keys in train_run_events_clean"
assert altered == 0, f"{altered} genuine observations were altered"
assert orphans == 0, f"{orphans} cleaned rows do not resolve to a master record"
assert negative_lateness == 0, f"{negative_lateness} lateness values are negative"
assert unresolved == 0, f"{unresolved} trains have an endpoint that does not resolve to a station"

assert con.sql(RAW_FINGERPRINT_SQL).fetchall() == raw_before, "A raw table was modified"
print("OK  cleaned layer is consistent and the raw tables are unchanged")

┌──────────┬────────────┬────────────────┬──────────────────────┬─────────────┬───────────────────┬──────────────────────┐
│ raw_rows │ clean_rows │ duplicate_keys │ altered_observations │ orphan_rows │ negative_lateness │ unresolved_endpoints │
│  int64   │   int64    │     int64      │        int64         │    int64    │       int64       │        int64         │
├──────────┼────────────┼────────────────┼──────────────────────┼─────────────┼───────────────────┼──────────────────────┤
│     2460 │       2460 │              0 │                    0 │           0 │                 0 │                    0 │
└──────────┴────────────┴────────────────┴──────────────────────┴─────────────┴───────────────────┴──────────────────────┘

OK  cleaned layer is consistent and the raw tables are unchanged


### 5.3 Effect on the headline measures

Removing the placeholders changes the averages. Early running and large delays are kept, so only the population moves.

In [7]:
show("""
    SELECT 'raw' AS layer,
        COUNT(arrival_delay_min) AS arrival_obs,
        ROUND(AVG(arrival_delay_min), 2) AS mean_arrival_delay,
        COUNT(departure_delay_min) AS departure_obs,
        ROUND(AVG(departure_delay_min), 2) AS mean_departure_delay,
        ROUND(AVG(date_diff('minute', actual_arrival, actual_departure)), 2) AS mean_dwell
    FROM train_run_events
    UNION ALL
    SELECT 'clean',
        COUNT(arrival_delay_min),
        ROUND(AVG(arrival_delay_min), 2),
        COUNT(departure_delay_min),
        ROUND(AVG(departure_delay_min), 2),
        ROUND(AVG(actual_dwell_min), 2)
    FROM train_run_events_clean
""")

┌─────────┬─────────────┬────────────────────┬───────────────┬──────────────────────┬────────────┐
│  layer  │ arrival_obs │ mean_arrival_delay │ departure_obs │ mean_departure_delay │ mean_dwell │
│ varchar │    int64    │       double       │     int64     │        double        │   double   │
├─────────┼─────────────┼────────────────────┼───────────────┼──────────────────────┼────────────┤
│ raw     │        2460 │               2.23 │          2460 │                 3.85 │       5.33 │
│ clean   │        2034 │                2.7 │          2034 │                 4.05 │       6.07 │
└─────────┴─────────────┴────────────────────┴───────────────┴──────────────────────┴────────────┘



In [8]:
# flag counts
show("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE is_origin) AS origin_rows,
        COUNT(*) FILTER (WHERE is_terminus) AS terminus_rows,
        COUNT(*) FILTER (WHERE has_negative_dwell) AS negative_dwell_rows,
        COUNT(*) FILTER (WHERE arrival_delay_min < 0) AS early_arrivals,
        COUNT(*) FILTER (WHERE departure_delay_min < 0) AS early_departures,
        COUNT(*) FILTER (WHERE is_large_arrival_delay) AS large_arrival_delays,
        COUNT(*) FILTER (WHERE is_large_departure_delay) AS large_departure_delays
    FROM train_run_events_clean
""")

┌───────────┬─────────────┬───────────────┬─────────────────────┬────────────────┬──────────────────┬──────────────────────┬────────────────────────┐
│ row_count │ origin_rows │ terminus_rows │ negative_dwell_rows │ early_arrivals │ early_departures │ large_arrival_delays │ large_departure_delays │
│   int64   │    int64    │     int64     │        int64        │     int64      │      int64       │        int64         │         int64          │
├───────────┼─────────────┼───────────────┼─────────────────────┼────────────────┼──────────────────┼──────────────────────┼────────────────────────┤
│      2460 │         426 │           426 │                  13 │            586 │              308 │                   51 │                     74 │
└───────────┴─────────────┴───────────────┴─────────────────────┴────────────────┴──────────────────┴──────────────────────┴────────────────────────┘



## 6. Preview the cleaned tables

In [9]:
show("DESCRIBE train_run_events_clean")

┌───────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │ column_type │  null   │   key   │ default │  extra  │
│          varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ service_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ service_date              │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ train_id                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ station_sequence          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ station_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_origin                 │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_terminus               │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ scheduled_arrival         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NU

In [10]:
# one run end to end, arrival should be null at the origin and departure at the terminus
show("""
    SELECT
        service_id, station_sequence, station_id, is_origin, is_terminus,
        actual_arrival, arrival_delay_min, actual_departure, departure_delay_min,
        actual_dwell_min, has_negative_dwell
    FROM train_run_events_clean
    WHERE service_id = (SELECT MIN(service_id) FROM train_run_events_clean)
    ORDER BY station_sequence
""")

┌────────────┬──────────────────┬────────────┬───────────┬─────────────┬─────────────────────┬───────────────────┬─────────────────────┬─────────────────────┬──────────────────┬────────────────────┐
│ service_id │ station_sequence │ station_id │ is_origin │ is_terminus │   actual_arrival    │ arrival_delay_min │  actual_departure   │ departure_delay_min │ actual_dwell_min │ has_negative_dwell │
│  varchar   │      int64       │  varchar   │  boolean  │   boolean   │      timestamp      │       int64       │      timestamp      │        int64        │      int64       │      boolean       │
├────────────┼──────────────────┼────────────┼───────────┼─────────────┼─────────────────────┼───────────────────┼─────────────────────┼─────────────────────┼──────────────────┼────────────────────┤
│ RUN000001  │                1 │ ST001      │ true      │ false       │ NULL                │              NULL │ 2026-08-01 07:02:00 │                   2 │             NULL │ false              │
│ RUN

In [11]:
show("SELECT * EXCLUDE (route_note) FROM train_master_clean ORDER BY train_id")

┌──────────┬──────────────────────┬───────────────────┬────────────────┬────────────────────────┬───┬───────────────┬───────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────────┐
│ train_id │      train_name      │ origin_station_id │ origin_station │ destination_station_id │ … │ stops_per_run │ runs_operated │ first_service_date │ last_service_date │ days_without_a_run │ non_operating_weekday │
│ varchar  │       varchar        │      varchar      │    varchar     │        varchar         │ … │     int64     │     int64     │        date        │       date        │       int64        │        varchar        │
├──────────┼──────────────────────┼───────────────────┼────────────────┼────────────────────────┼───┼───────────────┼───────────────┼────────────────────┼───────────────────┼────────────────────┼───────────────────────┤
│ TR001    │ Subarna Express      │ ST001             │ Dhaka          │ ST011                  │ … │             5 │   

In [12]:
show("SELECT * FROM station_master_clean ORDER BY station_id")

┌────────────┬─────────────────────────┬─────────┬─────────────┬───────────┐
│ station_id │      station_name       │ region  │ stop_events │ is_served │
│  varchar   │         varchar         │ varchar │    int64    │  boolean  │
├────────────┼─────────────────────────┼─────────┼─────────────┼───────────┤
│ ST001      │ Dhaka                   │ Central │         426 │ true      │
│ ST002      │ Biman Bandar            │ Central │         207 │ true      │
│ ST003      │ Joydebpur               │ Central │         258 │ true      │
│ ST004      │ Tangail                 │ West    │         168 │ true      │
│ ST005      │ Ibrahimabad             │ West    │          45 │ true      │
│ ST006      │ Bangabandhu Bridge East │ West    │           0 │ false     │
│ ST007      │ Santahar                │ North   │         123 │ true      │
│ ST008      │ Bogra                   │ North   │          84 │ true      │
│ ST009      │ Joypurhat               │ North   │          39 │ true      │

## 7. Summary

All ten corrections from notebook 02 are applied in the cleaned layer and verified in section 5. Row count and keys are preserved, and no genuine observation was changed.

- Use `train_run_events_clean` for delay and dwell measures. Origin arrivals, terminus departures and impossible dwell values are `NULL`, so plain aggregates are correct.
- Use `arrival_late_min` and `departure_late_min` for lateness, and the signed delay columns for average delay.
- Use `runs_operated` in `train_master_clean` as the denominator for per-train rates.
- Join to the cleaned master tables for names.

Not fixable here: early arrivals are floored at -2 minutes in the source, which should be stated in any report.

In [13]:
# release the lock so the other notebooks can open the db
con.close()
print("Connection closed.")

Connection closed.
